# Experiment 9 — Needham-Schroeder Authentication Protocol
Simulates symmetric-key NS protocol · Replay attack · Denning-Sacco fix

## Symmetric Cipher (RC4, inlined)

In [ ]:
import secrets, time

def _ksa(key):
    S=list(range(256)); j=0
    for i in range(256): j=(j+S[i]+key[i%len(key)])%256; S[i],S[j]=S[j],S[i]
    return S
def _prga(S,n):
    S=S[:]; i=j=0; out=[]
    for _ in range(n):
        i=(i+1)%256; j=(j+S[i])%256; S[i],S[j]=S[j],S[i]; out.append(S[(S[i]+S[j])%256])
    return out
def sym_enc(key,pt): S=_ksa(key); ks=_prga(S,len(pt)); return bytes(a^b for a,b in zip(pt,ks))
def sym_dec(key,ct): return sym_enc(key,ct)

def gen_key(): return secrets.token_bytes(16)
print("RC4 symmetric cipher loaded ✓")


## KDC & Principal Setup

In [ ]:
class KDC:
    def __init__(self): self.keys={}
    def register(self,name):
        k=gen_key(); self.keys[name]=k; return k
    def issue_ticket(self,alice,bob):
        Ka=self.keys[alice]; Kb=self.keys[bob]; Kab=gen_key()
        ticket=sym_enc(Kb, Kab+alice.encode().ljust(16))
        outer =sym_enc(Ka, Kab+bob.encode().ljust(16)+ticket)
        return outer, Kab

kdc=KDC()
Ka=kdc.register("Alice")
Kb=kdc.register("Bob")
print("KDC and principals initialized ✓")


## Protocol Execution

In [ ]:
Na=secrets.token_bytes(8)
print(f"Step 1 — Alice→KDC : (Alice, Bob, Na={Na.hex()})")

bundle, Kab_real = kdc.issue_ticket("Alice","Bob")
print(f"Step 2 — KDC→Alice : encrypted bundle ({len(bundle)} bytes)")

outer = sym_dec(Ka, bundle)
Kab_alice = outer[:16]; ticket = outer[32:]
print(f"       Alice extracts Kab={Kab_alice.hex()}")

Nb=secrets.token_bytes(8)
msg3=ticket+sym_enc(Kab_alice,Nb)
print(f"Step 3 — Alice→Bob : ticket + {{Nb}}_Kab")

tkt_plain=sym_dec(Kb,ticket); Kab_bob=tkt_plain[:16]
Nb_recv=sym_dec(Kab_bob,msg3[len(ticket):])
print(f"       Bob decrypts ticket, Kab={Kab_bob.hex()}, Nb={Nb_recv.hex()}")

Nb1=(int.from_bytes(Nb_recv,'big')-1).to_bytes(8,'big')
msg4=sym_enc(Kab_bob,Nb1)
print(f"Step 4 — Bob→Alice : {{Nb-1}}_Kab")

recv=sym_dec(Kab_alice,msg4)
expected=(int.from_bytes(Nb,'big')-1).to_bytes(8,'big')
ok=recv==expected
print(f"Alice verifies Nb-1: {'✓ OK' if ok else '✗ FAILED'}")
assert ok and Kab_alice==Kab_bob
print(f"\nSession key: {Kab_alice.hex()}")
print("✓ Mutual authentication complete, shared session key established")


## Replay Attack Demonstration

In [ ]:
print("=== Replay Attack on Original NS Protocol ===")
print()
print("Attacker Eve recorded an old session ticket for Bob: {K_old, Alice}_Kb")
print("Eve replays Step 3 using the OLD ticket (with compromised K_old).")
print("Bob decrypts the ticket, gets K_old, sends Nb encrypted with K_old.")
print("Eve (who knows K_old) completes the handshake.")
print("Bob believes he's talking to Alice — but it's Eve!")
print()
print("Root cause: ticket has NO freshness indicator.")


## Denning-Sacco Fix — Timestamps

In [ ]:
T=time.time(); MAX_SKEW=300
T_fresh=T-30; T_replay=T-600

print("Fix: KDC embeds timestamp T in ticket: {Kab, A, T}_Kb")
print(f"Bob rejects if |T_now - T| > {MAX_SKEW}s")
print()
print(f"Current time       : {T:.0f}")
print(f"Fresh ticket  age  : {T-T_fresh:.0f}s → {'ACCEPT ✓' if abs(T-T_fresh)<MAX_SKEW else 'REJECT ✗'}")
print(f"Replay ticket age  : {T-T_replay:.0f}s → {'ACCEPT ✓' if abs(T-T_replay)<MAX_SKEW else 'REJECT ✗'}")
print()
print("✓ Replay rejected — timestamp too old")
print("Requirement: NTP clock synchronisation between parties")


## Security Summary

In [ ]:
print(f"{'Attack':<20} | {'Original NS':<14} | NS + Timestamps")
print('-'*55)
rows=[("Eavesdropping","Resistant","Resistant"),
      ("Replay attack","Vulnerable ✗","Resistant ✓"),
      ("MITM (via KDC)","Resistant","Resistant"),
      ("Clock sync needed","No","Yes")]
for a,o,t in rows:
    print(f"  {a:<18} | {o:<14} | {t}")
